# 06 Backtesting

Backtest the walk-forward signal stream and compare it with a simple buy-and-hold benchmark.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

import pandas as pd
from backtester import BacktestConfig, backtest_signals
from data_loader import load_ohlcv_csv
from performance import summarize_performance
from utils import project_path

In [ ]:
prices = load_ohlcv_csv(project_path("data", "raw", "SPY.csv"))
predictions = pd.read_csv(project_path("data", "processed", "SPY_predictions_xgboost.csv"), index_col=0, parse_dates=True)

backtest = backtest_signals(prices, predictions["signal"], BacktestConfig())
strategy_metrics = summarize_performance(backtest["strategy_return"], backtest["equity_curve"])

buy_hold_returns = prices["close"].pct_change().fillna(0.0)
buy_hold_equity = 100_000 * (1 + buy_hold_returns).cumprod()
buy_hold_metrics = summarize_performance(buy_hold_returns, buy_hold_equity)

backtest.to_csv(project_path("data", "processed", "SPY_backtest.csv"))
pd.DataFrame({"strategy": strategy_metrics, "buy_and_hold": buy_hold_metrics})